# 동결 분류기 + KNN 게이팅 — 원문제 모범답안2 코드 그대로 (Colab)

**연습 기법** (IOAI 2024 *onsite* **Help BOBAI** *모범답안2*(공식 우수풀이, macro-F1 ≈0.79) 와 동일): 이미 배포된
**분류기를 재학습하지 않고**(파라미터 학습 금지), 신규 클래스는 **KNN(거리) 게이팅**으로만 추가한다. **이 노트북
코드는 BOBAI 모범답안2 골격을 그대로** 따른다:

| Help BOBAI 모범답안2 | 이 노트북 |
|---|---|
| **동결 base 분류기**(`base_classifier.pth`, 재학습 금지) | 동결 LogReg(0~4) — *제공 pth 없어 학습 후 동결* |
| `KNeighborsClassifier(n_neighbors=1, weights='distance', metric='cosine')` | 동일 (그대로) |
| `gate_labels`(0=기존, 1·2=신규) → `predict_batch`(`which+4 if which else 동결예측`) | 동일(0~4 기존 + 5~9 신규로 일반화) |

**시나리오**: 숫자 **0~4** 는 *배포된* 5-way 분류기 담당(동결). **5~9** 는 나중에 추가된 신규 — 라벨은 있지만 배포
모델을 못 고쳐 **KNN 으로 신규 여부를 게이팅**해 배정(BOBAI 의 `which+4` 로직 일반화).

> ⚙️ T4 수 분(scikit-learn 만). digit-recognizer 엔 *배포 pth* 가 없어 0~4 로 LogReg 를 학습→동결해 *배포 모델* 을 모사.
> ⚠️ API 토큰 평문 — 외부 공유 금지. (08·15·18 과 같은 digit-recognizer)

## 0. 설치 · 자격증명

In [1]:
import sys, subprocess
for pkg in ["kaggle", "numpy", "pandas", "scikit-learn"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("준비 완료")

준비 완료


## 1. Kaggle 에서 Digit Recognizer 다운로드 & 기존(0~4)/신규(5~9) 분할

In [2]:
import glob, zipfile, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("digit-recognizer", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
train = pd.read_csv(os.path.join(WORK_DIR, "train.csv")); test = pd.read_csv(os.path.join(WORK_DIR, "test.csv"))
y = train["label"].to_numpy()
X = train.drop(columns=["label"]).to_numpy(dtype=np.float32) / 255.0
X_test = test.to_numpy(dtype=np.float32) / 255.0
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("train:", X_tr.shape, "| val:", X_val.shape, "| 기존 0~4, 신규 5~9")

100%|██████████| 15.3M/15.3M [00:00<00:00, 81.9MB/s]



train: (33600, 784) | val: (8400, 784) | 기존 0~4, 신규 5~9


## 2. 배포된 5-way 분류기 학습 후 **동결** (BOBAI 의 base_classifier.pth 역할)
기존 클래스 0~4 로만 분류기를 만들고, 이후 파라미터를 바꾸지 않는다(BOBAI 제약).

In [3]:
from sklearn.linear_model import LogisticRegression
mask_old = y_tr < 5                       # 기존 클래스 데이터만
base_clf = LogisticRegression(max_iter=200, n_jobs=-1)
base_clf.fit(X_tr[mask_old], y_tr[mask_old])
print("동결 5-way 분류기 학습 완료 (클래스:", sorted(set(y_tr[mask_old])), ") — 이후 재학습/수정 금지")

동결 5-way 분류기 학습 완료 (클래스: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)] ) — 이후 재학습/수정 금지


## 3. KNN 게이트 (cosine, k=1) 학습 — 신규(5~9) 여부 판별
`gate` 라벨(0=기존 0~4, 1~5=신규 5~9)을 만들고, 참조용 **12,000개 샘플**로 `KNeighborsClassifier(n_neighbors=1, weights='distance', metric='cosine')` 를 학습한다 — BOBAI 모범답안2 의 게이트와 동일.

In [12]:
import os, zipfile, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
gate=np.where(y_tr<5,0,y_tr-4)
idx=np.random.RandomState(0).permutation(len(X_tr))[:12000]
knn=KNeighborsClassifier(n_neighbors=1, weights="distance", metric="cosine").fit(X_tr[idx], gate[idx])

## 4. 추론 함수 — 게이트가 신규면 `which+4`, 기존이면 동결 분류기 (모범답안2의 predict_batch)
`predict(X)`: cosine-KNN 게이트가 신규(>0)면 `which+4`(→5~9), 아니면 **동결 `base_clf`** 예측(0~4). BOBAI 모범답안2 의 `predict_batch`(`which+4 if which else 동결`) 로직 그대로.

In [13]:
def predict(X_in):
    which, old = knn.predict(X_in), base_clf.predict(X_in)
    return np.where(which > 0, which + 4, old)

## 5. 캐글 제출 파일 생성 (`test.csv` → `submission.csv`)

In [10]:
test_pred = predict(X_test)
submission_path = os.path.join(WORK_DIR, "submission.csv")
pd.DataFrame({"ImageId": np.arange(1, len(test_pred) + 1), "Label": test_pred}).to_csv(submission_path, index=False)
print("Saved:", submission_path)
pd.read_csv(submission_path).head()

Saved: /content/submission.csv


,ImageId,Label
0,1,2
1,2,0
2,3,9
3,4,9
4,5,3


In [11]:
try:
    from google.colab import files; files.download(submission_path)
except Exception:
    print("submission.csv 위치:", submission_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 🏁 제출하기
`submission.csv` 를 [Digit Recognizer](https://www.kaggle.com/competitions/digit-recognizer) 에 제출.

BOBAI **모범답안2**(cosine-KNN 게이팅) 골격(**동결 분류기 + `KNeighborsClassifier(cosine,k=1)` 게이트 + `predict_batch`
(`which+4 if which else 동결`)**)을 그대로 옮겼다. 핵심은 *배포 모델을 재학습하지 않고 거리(KNN)로만 신규
클래스를 추가*. 더: KNN `k`·참조셋 크기, 동결 분류기의 임베딩 품질(여기선 raw 픽셀).